# Module 03 — Autograd and Computational Graphs

**Prerequisites:** Module 02 (Tensor Operations & Broadcasting)
**Time:** ~60 minutes

## Learning Objectives

- Explain what a gradient is and why training a neural network needs one.
- Explain what a computational graph is and how PyTorch builds one automatically.
- Use `requires_grad`, `.backward()`, and `.grad` to compute gradients.
- Explain why gradients accumulate by default, and why `zero_grad()` exists.
- Use `torch.no_grad()` correctly, and explain what it turns off.


## Why Do We Need Gradients?

Training a neural network means: gradually adjusting its parameters (weights) so its predictions get better. "Better" is measured by a **loss function** — a single number that's high when predictions are bad and low when they're good.

The question training needs answered, over and over, is: *if I nudge this particular weight up slightly, does the loss go up or down, and by how much?* That sensitivity — "how much does the loss change per tiny change in this weight" — is exactly what a **derivative** (a **gradient**, once you have many weights) tells you.

Once you know the gradient of the loss with respect to every weight, you can nudge every weight in the direction that *decreases* the loss. Repeat thousands of times, and the model gets better. This process is called **gradient descent**, and it's the algorithm underneath essentially every neural network ever trained.

The problem: a real model might have millions of weights, connected through dozens of layers of arithmetic. Computing every one of those derivatives by hand (as you'd do with pen-and-paper calculus, using the chain rule repeatedly) is completely impractical. **Autograd is PyTorch's system for computing all of these derivatives automatically**, and it's arguably the single most important feature PyTorch provides.


In [1]:
import torch

## A Tiny Example, By Hand First

Let's use the smallest possible example so the calculus is trivial, and compare "doing it by hand" to "letting autograd do it."

Suppose $z = x^2$. From calculus, $\frac{dz}{dx} = 2x$. If $x = 3$, then $\frac{dz}{dx} = 6$.


In [2]:
x = torch.tensor(3.0, requires_grad=True)   # requires_grad=True: "track operations on this tensor"
z = x ** 2

z.backward()   # compute d(z)/d(x) for every tensor that requires_grad, working backward from z

print("x.grad:", x.grad)   # should be 2 * 3 = 6, matching the hand calculation


x.grad: tensor(6.)


That matched our hand calculation: `2 * 3 = 6`. Now let's see what's actually happening.

**`requires_grad=True`** tells PyTorch: "remember every operation applied to this tensor, because I'll want gradients with respect to it later." Tensors created this way (or derived from such tensors) are part of a **computational graph** — a record of every operation, connecting inputs to outputs.

**`z.backward()`** walks that graph *backward*, starting from `z`, applying the chain rule at every step, until it reaches every `requires_grad=True` tensor that contributed to `z`. At each one, it stores the computed derivative in that tensor's `.grad` attribute.

This is called **tape-based** (or "reverse-mode") automatic differentiation: PyTorch records a "tape" of operations during the forward pass, then replays it backward to compute gradients.


### A Slightly Bigger Example

Let's verify the chain rule works for a compound expression: $z = x^2 + 3x$, so $\frac{dz}{dx} = 2x + 3$. At $x=3$: $2(3)+3=9$.


In [3]:
x = torch.tensor(3.0, requires_grad=True)
z = x ** 2 + 3 * x

z.backward()
print("x.grad:", x.grad)   # expect 9


x.grad: tensor(9.)


### 🔮 Predict before you run

For a *vector* input `x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)` and `z = (x ** 2).sum()`, what do you expect `x.grad` to be? (Hint: the derivative of $\sum x_i^2$ with respect to each $x_i$ is $2x_i$ — apply this elementwise.)


In [4]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
z = (x ** 2).sum()
z.backward()
print("x.grad:", x.grad)   # expect [2, 4, 6]


x.grad: tensor([2., 4., 6.])


`.backward()` requires the tensor you call it on (`z` here) to be a **scalar** (a single number) — that's why we called `.sum()` first. This makes sense conceptually: "the gradient of a vector with respect to another vector" is more complex than a single number's worth of information, and in the overwhelming majority of deep learning, the thing you call `.backward()` on is the loss — always a single scalar number.


## Three Rules of Autograd

1. **Only leaf tensors with `requires_grad=True` accumulate gradients into `.grad`.** A "leaf" tensor is one you created directly (not the result of an operation on other tracked tensors). Model parameters (`nn.Linear`'s weights, for example) are leaf tensors with `requires_grad=True` set automatically — you'll see this in the next module.
2. **Gradients accumulate (add up) by default across multiple `.backward()` calls**, rather than being overwritten. This is intentional — it supports advanced use cases — but it means you must explicitly reset gradients to zero before each new backward pass in a training loop, or old gradients will corrupt the new computation.
3. **`torch.no_grad()` disables gradient tracking** for a block of code — used whenever you don't need gradients (evaluation, inference), because tracking has both a memory and compute cost.

Let's see rule 2 in action, since it's the one that causes real bugs.


In [5]:
x = torch.tensor(2.0, requires_grad=True)

z1 = x ** 2
z1.backward()
print("after first backward:", x.grad)   # 2*2 = 4

z2 = x ** 2
z2.backward()
print("after second backward (no zeroing!):", x.grad)   # NOT 4 again -- it's 4+4=8!


after first backward: tensor(4.)
after second backward (no zeroing!): tensor(8.)


This is exactly why every training loop you'll write contains `optimizer.zero_grad()` at the start of each iteration — without it, gradients from *every previous batch* silently pile up on top of each other, and the model updates itself using wrong, inflated gradients. Let's fix the example above:


In [6]:
x = torch.tensor(2.0, requires_grad=True)

z1 = x ** 2
z1.backward()
print("after first backward:", x.grad)

x.grad.zero_()   # reset gradient to 0 before the next backward pass
z2 = x ** 2
z2.backward()
print("after zeroing + second backward:", x.grad)   # correctly 4 again


after first backward: tensor(4.)
after zeroing + second backward: tensor(4.)


## `torch.no_grad()`: Turning Tracking Off

Every operation on a `requires_grad=True` tensor costs extra memory (to store the computational graph) and extra compute (to enable `.backward()` later). During evaluation or inference — when you'll never call `.backward()` — this tracking is pure waste.


In [7]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2   # this operation is NOT tracked

print("y.requires_grad:", y.requires_grad)   # False, even though x requires grad!

# Trying to call .backward() on y would raise an error, since no graph was recorded:
try:
    y.backward()
except RuntimeError as e:
    print("Error:", e)


y.requires_grad: False
Error: element 0 of tensors does not require grad and does not have a grad_fn


You'll see `with torch.no_grad():` wrapped around every evaluation loop and every inference call in this course, for exactly this reason.


## 🐛 Debugging Challenge

The code below is supposed to compute the gradient of `loss` with respect to `w`, but it fails. Find the bug before reading the explanation.


In [8]:
w = torch.tensor(5.0)   # note: no requires_grad!
target = torch.tensor(10.0)

prediction = w * 2
loss = (prediction - target) ** 2

try:
    loss.backward()
    print(w.grad)
except RuntimeError as e:
    print("Error:", e)


Error: element 0 of tensors does not require grad and does not have a grad_fn


**Diagnosis:** `w` was created without `requires_grad=True`, so PyTorch never tracked any operations involving it — there's no graph connecting `loss` back to `w`, so there's nothing to walk backward through.

**Fix:** set `requires_grad=True` when creating `w`.


In [8]:
w = torch.tensor(5.0, requires_grad=True)
target = torch.tensor(10.0)

prediction = w * 2
loss = (prediction - target) ** 2
loss.backward()
print(w.grad)   # d(loss)/dw = 2*(2w - 10)*2 = 4*(2*5-10) = 0 here, since prediction already equals target


tensor(0.)


## Where This Is Going

In the next module, you'll build a tiny model manually — a single linear layer computed as `y = x @ W + b` — and use exactly this machinery (`requires_grad`, `.backward()`, `.grad`) to train it with gradient descent, *before* introducing `nn.Linear` and `nn.Module`. The goal is for you to see, concretely, that PyTorch's high-level abstractions are just convenient wrappers around the mechanics you just learned.


## Exercises

🟢 **Beginner:** Create `x = torch.tensor(4.0, requires_grad=True)`. Compute `y = 3 * x + 1`, call `.backward()`, and check that `x.grad` equals 3 (the coefficient — since $y = 3x + 1$ has constant slope 3).

🟡 **Intermediate:** Create `x = torch.tensor(2.0, requires_grad=True)`. In a loop that runs 3 times: compute `y = x ** 2`, call `.backward()`, and print `x.grad` *without* zeroing it between iterations. Confirm the values are `4, 8, 12` (accumulating), then repeat the loop but zero the gradient each iteration and confirm you instead get `4, 4, 4`.

🔴 **Challenge:** Implement one manual step of gradient descent on the function $f(x) = (x-5)^2$, which has its minimum at $x=5$. Starting from `x = torch.tensor(0.0, requires_grad=True)` and a learning rate of `0.1`: compute the loss, call `.backward()`, then update `x` using `with torch.no_grad(): x -= 0.1 * x.grad` (the `no_grad()` block is required here — updating a `requires_grad=True` tensor in place while tracked would itself be tracked, which is not what we want). Run this update 20 times in a loop (remembering to zero the gradient each iteration) and print `x` at the end — it should be noticeably closer to 5.


In [10]:
# Space for your exercise solutions



## Common Mistakes

- **Forgetting `requires_grad=True`** on a tensor you intend to compute gradients for, then being confused when `.grad` is `None`.
- **Forgetting to zero gradients** between training steps, causing gradients to silently accumulate across batches.
- **Calling `.backward()` on a non-scalar tensor** without reducing it to a single number first (usually via `.sum()` or `.mean()`).
- **Forgetting `torch.no_grad()`** during evaluation/inference, wasting memory and compute (and in the exercise above, forgetting it during a manual parameter update, which would incorrectly try to track the update itself).

## Mental Model

Think of every `requires_grad=True` tensor as carrying an invisible "how did I get here?" tag. As you compute `y = x**2`, then `z = y + 3`, PyTorch is silently building a chain: `x → y → z`. Calling `z.backward()` walks that chain backward, applying the chain rule at every link, and drops the final answer into `x.grad`. `torch.no_grad()` is a way of saying "don't bother tagging anything in here — I won't need the history."

## Key Takeaways

- A gradient tells you how much a small change in one number affects another — training uses this to know which direction to adjust weights.
- `requires_grad=True` + `.backward()` is PyTorch's automatic differentiation system, replacing hand-derived calculus.
- Gradients accumulate by default — always zero them before the next backward pass in a training loop.
- `torch.no_grad()` disables tracking for evaluation/inference, saving memory and compute.

## What's Next

**Module 04 — Building Neural Networks: From Scratch to `nn.Module`** uses everything from this notebook to build, train, and then refactor a tiny model — first with raw tensors and autograd, then using PyTorch's `nn.Linear` and `nn.Module` abstractions.

## Checklist

- [ ] I can explain in my own words why training needs gradients
- [ ] I can compute a gradient with `requires_grad=True` and `.backward()` and verify it against hand calculus
- [ ] I understand why gradients accumulate and why `zero_grad()`/`.grad.zero_()` is necessary
- [ ] I know when and why to use `torch.no_grad()`
